# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/02017711723iot-dotcom/Flyrank_Internship_1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os
import subprocess

REPO_URL = "https://github.com/02017711723iot-dotcom/Flyrank_Internship_1"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

print("Repository ready!")
print(os.listdir(REPO_DIR))

Repository ready!
['scripts', '.github', 'AGENTS.md', 'submission', 'outputs', '.gitignore', 'SETUP.md', 'skills', 'GUIDE.md', 'notebooks', 'requirements.txt', 'LICENSE', 'README.md', 'CLAUDE.md', '.git', 'data', 'DATA_USE.md', 'docs', 'work']


In [10]:
import pandas as pd

DATA_PATH = "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset loaded successfully!
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
# Show all columns in the dataset

print("Number of columns:", len(df.columns))
print("\nColumns:")

for i, column in enumerate(df.columns, 1):
    print(i, "-", column)

Number of columns: 44

Columns:
1 - content_id
2 - client_id
3 - search_volume
4 - competition
5 - competition_level
6 - cpc
7 - content_type
8 - main_intent
9 - word_count
10 - char_count
11 - provider_used
12 - model_used
13 - impressions_90d
14 - clicks_90d
15 - pageviews_90d
16 - sessions_90d
17 - users_90d
18 - engaged_sessions_90d
19 - ai_sessions_90d
20 - scroll_events_90d
21 - days_with_impressions
22 - days_with_sessions
23 - impressions_last_30d
24 - clicks_last_30d
25 - sessions_last_30d
26 - impressions_prev_30d
27 - clicks_prev_30d
28 - sessions_prev_30d
29 - content_age_days
30 - age_tier
31 - age_tier_order
32 - days_since_last_update
33 - freshness_tier
34 - word_count_tier
35 - char_count_tier
36 - ctr
37 - avg_position
38 - engagement_rate
39 - scroll_rate
40 - ai_traffic_pct
41 - impression_tier
42 - position_tier
43 - trend_direction
44 - trend_pct


In [12]:
# Check the columns needed for our Refresh / Content Opportunity lane

required_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction"
]

print("Checking required columns...\n")

for column in required_columns:
    if column in df.columns:
        print("FOUND  :", column)
    else:
        print("MISSING:", column)

Checking required columns...

FOUND  : content_age_days
FOUND  : days_since_last_update
FOUND  : impressions_90d
FOUND  : avg_position
FOUND  : ctr
FOUND  : word_count
FOUND  : trend_direction


In [13]:
# STEP 3 — CREATE TARGET + CLIENT-GROUPED TRAIN/TEST SPLIT

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Make a separate copy
data = df.copy()


# Create the target
# 1 = declining, 0 = not declining

data["is_declining_label"] = (
    data["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Features available before the outcome
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Clean numeric values
data[features] = (
    data[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Remove rows without a client ID
data = data.dropna(subset=["client_id"]).reset_index(drop=True)

# Client-grouped split

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        data,
        data["is_declining_label"],
        groups=data["client_id"]
    )
)

train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

# Display split information

print("========== DATA SPLIT ==========")

print("Total rows:", len(data))

print("\nTraining rows:", len(train))
print("Testing rows :", len(test))

print("\nTraining clients:", train["client_id"].nunique())
print("Testing clients :", test["client_id"].nunique())

# ------------------------------------------------------------
# Check for client leakage
# ------------------------------------------------------------

train_clients = set(train["client_id"])
test_clients = set(test["client_id"])

client_overlap = train_clients.intersection(test_clients)

print("\nClients appearing in BOTH sets:", len(client_overlap))

if len(client_overlap) == 0:
    print("✓ Split is clean — no client leakage.")
else:
    print("✗ WARNING — client leakage detected!")

# ------------------------------------------------------------
# Target distribution
# ------------------------------------------------------------

print("\n========== TARGET ==========")

print(
    "Training declining rate:",
    round(train["is_declining_label"].mean(), 3)
)

print(
    "Testing declining rate :",
    round(test["is_declining_label"].mean(), 3)
)

# Show sample

print("\n========== TRAINING SAMPLE ==========")

display(
    train[
        [
            "client_id",
            "content_id",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "word_count",
            "trend_direction",
            "is_declining_label"
        ]
    ].head(10)
)

========== DATA SPLIT ==========
Total rows: 30000

Training rows: 23837
Testing rows : 6163

Training clients: 25
Testing clients : 7

Clients appearing in BOTH sets: 0
✓ Split is clean — no client leakage.

========== TARGET ==========
Training declining rate: 0.55
Testing declining rate : 0.511

========== TRAINING SAMPLE ==========


,client_id,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
2,client_7f2253d7e2,content_9aa793d4d895,141,20,12581,36.5,0.09,3515.0,down,1
3,client_19581e27de,content_331d6c4de07b,463,22,11751,6.2,0.49,0.0,stable,0
4,client_3fdba35f04,content_d99b7a2d90ca,263,14,19140,44.0,0.13,2803.0,down,1
6,client_8722616204,content_9a34b442b552,90,20,20,7.0,0.00,3059.0,down,1
7,client_19581e27de,content_a63219c6e95a,445,22,1724,21.2,0.06,0.0,stable,0
8,client_6208ef0f77,content_5e6c160719bc,90,20,32574,46.0,0.09,3807.0,down,1
9,client_19581e27de,content_c27558df2b0c,257,104,1240,4.9,0.16,0.0,down,1
10,client_19581e27de,content_d8ee6cc6d642,329,104,20919,2.2,1.55,0.0,stable,0
11,client_d4735e3a26,content_5a3e876cf7f7,312,20,1,0.0,0.00,776.0,new,0
12,client_6208ef0f77,content_42fb2cad9ecf,124,104,7228,5.6,1.76,3969.0,up,0


In [14]:
# STEP 4 — VERIFY MODEL FEATURES

print("Features used by the model:")

for i, feature in enumerate(features, 1):
    print(i, "-", feature)

print("\nFeatures deliberately excluded:")

print("- trend_direction → used to create the target")
print("- trend_pct       → directly related to the target/outcome")
print("- is_declining_label → the target itself")

print("\nFeature matrix shape:")
print("Training:", train[features].shape)
print("Testing :", test[features].shape)

Features used by the model:
1 - content_age_days
2 - days_since_last_update
3 - impressions_90d
4 - avg_position
5 - ctr
6 - word_count

Features deliberately excluded:
- trend_direction → used to create the target
- trend_pct       → directly related to the target/outcome
- is_declining_label → the target itself

Feature matrix shape:
Training: (23837, 6)
Testing : (6163, 6)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a decision tree classifier for the Refresh / Content Opportunity Scoring lane.

The model will estimate the probability that a page is declining based on signals that are available before the decision, such as content age, update recency, impressions, average position, CTR, and word count.

A decision tree is useful here because its decisions can be inspected and explained as simple rules. This is important for a content team because the output is intended to create a ranked review queue, not to make an automatic refresh decision.

I will compare the model against my Week-4 hand-written baseline using the same evaluation metric and validation split.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Setup
import os
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import precision_score

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [16]:
# CAPSTONE — SECTION 1
# Load the FlyRank starter dataset

import os
import pandas as pd
import numpy as np

# Exact dataset path from the cloned FlyRank repository
DATA_PATH = "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

# Check that the file exists
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Make sure the FlyRank repository has been cloned in this Colab session."
    )

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# Create the target
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Declining pages:", df["is_declining_label"].sum())
print(
    "Declining rate:",
    round(df["is_declining_label"].mean(), 3)
)

display(df.head())

Dataset loaded successfully!
Rows: 30000
Columns: 44
Declining pages: 16262
Declining rate: 0.542


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a stratified 80/20 train-test split.

The model will be trained on 80% of the pages and evaluated on the remaining 20%. Stratification keeps the proportion of declining and non-declining pages similar in both sets.

This split is appropriate for this first capstone comparison because the starter dataset does not provide a clean chronological observation structure for every page that would allow a reliable future-window split from the available columns.

The test set will be kept separate from model fitting and will be used to compare the model's ranking performance against the same baseline approach.

The main evaluation metric will be Precision@50 because the content team needs a small ranked review queue rather than predictions for every page.

In [17]:
# CAPSTONE — SECTION 2
# Train / Test Split

from sklearn.model_selection import train_test_split

# Features available before the outcome
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Create X and y
X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"]

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Split completed successfully!")
print("--------------------------------")
print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

print("\nTraining declining rate:",
      round(y_train.mean(), 3))

print("Testing declining rate :",
      round(y_test.mean(), 3))

Split completed successfully!
--------------------------------
Training rows: 24000
Testing rows : 6000

Training declining rate: 0.542
Testing declining rate : 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train a decision tree using the training set and evaluate it on the held-out test set.

I will compare its Precision@50 with my Week-4 baseline using exactly the same test pages. This makes the comparison fair because both approaches are evaluated on the same data and with the same metric.

The baseline uses the hand-written stale-and-visible rule, while the decision tree learns how the available pre-decision signals relate to the declining label.

The purpose of this comparison is not simply to find the highest score. I want to determine whether the learned model provides useful additional ranking signal compared with the simpler rule.

In [22]:
# CAPSTONE — SECTION 3
# Train Decision Tree + Compare with Week-4 Baseline

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

print("Starting Section 3...")
print("-" * 50)


# 1. Check that required variables exist
required_variables = [
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "df"
]

missing = [v for v in required_variables if v not in globals()]

if missing:
    raise NameError(
        "These variables are missing: "
        + ", ".join(missing)
        + ". Please run the previous sections first."
    )

print("Required variables found successfully.")


# 2. Make sure the target exists

if "is_declining_label" not in df.columns:

    if "trend_direction" in df.columns:
        df["is_declining_label"] = (
            df["trend_direction"]
            .astype(str)
            .str.lower()
            .eq("down")
            .astype(int)
        )

        print("Created is_declining_label from trend_direction.")

    else:
        raise ValueError(
            "Neither is_declining_label nor trend_direction "
            "exists in the dataset."
        )


# 3. Train Decision Tree

model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

print("Decision Tree trained successfully!")


# 4. Precision@K function

def precision_at_k(scores, labels, k=50):

    scores = np.asarray(scores)
    labels = np.asarray(labels)

    # Do not allow K to be larger than the dataset
    k = min(k, len(labels))

    # Rank highest scores first
    order = np.argsort(-scores)

    # Select top K
    top_k_labels = labels[order[:k]]

    # Precision@K
    return top_k_labels.mean()


# 5. Decision Tree predictions on TEST data

model_scores = model.predict_proba(X_test)[:, 1]

model_precision_50 = precision_at_k(
    model_scores,
    y_test,
    k=50
)

print(f"Decision Tree Precision@50: {model_precision_50:.3f}")


# 6. Get the corresponding TEST rows from original dataframe

if hasattr(X_test, "index"):

    test_data = df.loc[X_test.index].copy()

else:

    # Fallback if X_test is a NumPy array
    test_data = df.iloc[-len(X_test):].copy()


# 7. Week-4 Hand-Written Baseline
# Rule:
# - Page is stale if last update >= 180 days
# - Page is visible if impressions >= 500
# - Higher impressions give higher priority

stale = (
    test_data["days_since_last_update"] >= 180
).astype(int)

visible = (
    test_data["impressions_90d"] >= 500
).astype(int)

baseline_scores = (
    stale
    * visible
    * test_data["impressions_90d"]
)


# 8. Baseline Precision@50
baseline_precision_50 = precision_at_k(
    baseline_scores,
    test_data["is_declining_label"],
    k=50
)

print(f"Week-4 Baseline Precision@50: {baseline_precision_50:.3f}")


# 9. Compare Model vs Baseline
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Hand Rule",
        "Decision Tree"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

display(comparison)


# 10. Calculate improvement
difference = (
    model_precision_50
    - baseline_precision_50
)

print(
    f"Model - Baseline difference: {difference:+.3f}"
)



# 11. Simple interpretation
if difference > 0:

    print(
        "\nResult: The Decision Tree has higher Precision@50 "
        "than the Week-4 baseline on this test split."
    )

elif difference < 0:

    print(
        "\nResult: The Week-4 baseline has higher Precision@50 "
        "than the Decision Tree on this test split."
    )

else:

    print(
        "\nResult: The Decision Tree and Week-4 baseline "
        "have the same Precision@50 on this test split."
    )

Starting Section 3...
--------------------------------------------------
Required variables found successfully.
Decision Tree trained successfully!
Decision Tree Precision@50: 0.680
Week-4 Baseline Precision@50: 0.640


,Method,Precision@50
0,Week-4 Hand Rule,0.64
1,Decision Tree,0.68


Model - Baseline difference: +0.040

Result: The Decision Tree has higher Precision@50 than the Week-4 baseline on this test split.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I will inspect both false positives and false negatives rather than relying only on the Precision@50 result.

A false positive is a page the model ranks as likely declining when the observed label is not declining. A false negative is a declining page that the model gives a relatively low score.

I will also inspect the decision tree's feature importance to understand which available signals the model relies on most.

These errors matter because a wrong recommendation can waste the content team's review time, while missing a genuinely declining page can leave a potentially useful content opportunity unreviewed.

The model's output should therefore be treated as a prioritization aid, not as proof that a page needs a refresh.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# CAPSTONE — SECTION 4A
# Feature importance

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Decision Tree Feature Importance:")
display(feature_importance)

# CAPSTONE — SECTION 4B
# Error analysis

# Create a test-set analysis dataframe
error_analysis = df.loc[X_test.index].copy()

# Add model probability
error_analysis["model_score"] = model_scores

# Predicted class using 0.5 threshold
error_analysis["predicted_class"] = (
    error_analysis["model_score"] >= 0.5
).astype(int)

# False positives
false_positives = error_analysis[
    (error_analysis["predicted_class"] == 1) &
    (error_analysis["is_declining_label"] == 0)
].copy()

# False negatives
# Model says not declining, actual label is declining

false_negatives = error_analysis[
    (error_analysis["predicted_class"] == 0) &
    (error_analysis["is_declining_label"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))


columns_to_show = [
    "content_id",
    "model_score",
    "is_declining_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

print("\nExample false positives:")
display(
    false_positives[
        columns_to_show
    ].sort_values(
        "model_score",
        ascending=False
    ).head(10)
)

print("\nExample false negatives:")
display(
    false_negatives[
        columns_to_show
    ].sort_values(
        "model_score",
        ascending=True
    ).head(10)
)
# CAPSTONE — SECTION 4C
# Read the decision tree
from sklearn.tree import export_text

tree_rules = export_text(
    model,
    feature_names=features
)

print(tree_rules)

Decision Tree Feature Importance:


,feature,importance
0,impressions_90d,0.566098
1,content_age_days,0.268744
2,avg_position,0.090354
3,ctr,0.074804
4,days_since_last_update,0.000000
5,word_count,0.000000


False positives: 1467
False negatives: 743

Example false positives:


,content_id,model_score,is_declining_label,days_since_last_update,impressions_90d,avg_position,ctr,word_count
4489,content_9888334d39cf,0.639099,0,104,10003,8.6,0.06,NaN
17697,content_ea651db31344,0.639099,0,20,12,6.1,0.00,2620.0
9902,content_4ef406aa3516,0.639099,0,20,127,18.3,0.00,2892.0
26973,content_c4e573bc451c,0.639099,0,20,595,5.2,0.17,2602.0
14908,content_cd3dac79f7b2,0.639099,0,104,2084,6.1,0.14,NaN
7970,content_d5ab35d2f20a,0.639099,0,20,15,45.2,0.00,3068.0
13496,content_4429cbb5f7e7,0.639099,0,20,377,66.9,0.00,2419.0
3372,content_0e9f1dc33e0a,0.639099,0,20,10,8.1,0.00,2372.0
22486,content_67d351276f8e,0.639099,0,104,169,36.3,0.00,NaN
7884,content_76a3a969d5ad,0.639099,0,20,812,15.4,0.12,4318.0



Example false negatives:


,content_id,model_score,is_declining_label,days_since_last_update,impressions_90d,avg_position,ctr,word_count
7006,content_d964111653a1,0.005171,1,104,3,0.0,0.0,1453.0
11851,content_b0a5c92e100b,0.185753,1,20,1,11.0,0.0,NaN
14620,content_fbc10a3efc41,0.185753,1,20,3,2.7,0.0,2690.0
17896,content_b2f81797c4af,0.185753,1,22,5,63.8,0.0,NaN
5893,content_9d70410bbefe,0.185753,1,104,5,5.6,0.0,3919.0
24526,content_538fbe0b5c19,0.185753,1,103,3,25.7,0.0,1926.0
10445,content_a02644d1dd2d,0.185753,1,20,4,5.8,0.0,1132.0
26692,content_453d445896e5,0.185753,1,20,2,5.0,0.0,619.0
22691,content_6233516e7e4a,0.185753,1,8,3,6.7,0.0,4537.0
11608,content_a83ed1aa5716,0.185753,1,28,2,23.5,0.0,3436.0


|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.65
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.65
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 344.50
|   |   |--- ctr <= 0.30
|   |   |   |--- class: 1
|   |   |--- ctr >  0.30
|   |   |   |--- class: 1
|   |--- content_age_days >  344.50
|   |   |--- avg_position <= 28.35
|   |   |   |--- class: 0
|   |   |--- avg_position >  28.35
|   |   |   |--- class: 0



## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.